In [3]:
import random
import time
from dataclasses import dataclass, field

@dataclass
class Agente:
    id: int
    k_max: int = 10
    k: int = 0
    intentos: int = 0
    exitoso: bool = False
    ronda_exito: int = -1

def elegir_espera(k: int) -> int:
  ventana = max(1,2 ** k)
  return random.randint(0, ventana - 1)


def simular_canal(agentes: list['Agente'], verbose: bool = True) -> dict:
  ronda = 0
  pendientes = list(agentes)


  if verbose:
    print(f"{'='*55}")
    print(f"Iniciando simulacion con {len(agentes)} agente(s)")
    print(f"{'='*55}")

  while pendientes:
    ronda += 1

    esperas = {a.id: elegir_espera(a.k) for a in pendientes}
    intentan = [a for a in pendientes if esperas[a.id] == 0]

    for a in pendientes:
      a.intentos += 1

    if verbose:
      print(f"\nRonda {ronda:>3} | pendientes: {[a.id for a in pendientes]}")
      print(f"Esperas: {{a.id: esperas[a.id] for a in pendientes}}")
      print(f"Tranmiten: {[a.id for a in intentan]}", end = " ->")

    if len(intentan) == 1:
      ganador = intentan[0]
      ganador.exitoso = True
      ganador.ronda_exito = ronda
      pendientes.remove(ganador)
      if verbose:
        print(f"Agente {ganador.id} transmite con exito (k={ganador.k})")

    elif len(intentan) > 1:
      if verbose:
        print(f"x colision entre {[a.id for a in intentan]}")
      for a in intentan:
        if a.k < a.k_max:
          a.k += 1
        else:
          a.exitoso = False
          pendientes.remove(a)
          if verbose:
            print(f"Agente descartado")
    else:
      if verbose:
        print("(nadie transmite)")

    for a in pendientes:
      if esperas[a.id] > 0:
          esperas[a.id] -= 1


  exitosos = [a for a in agentes if a.exitoso]
  fallidos = [a for a in agentes if not a.exitoso]

  if verbose:
          print(f"\n{'='*55}")
          print(f"  Simulación completada en {ronda} rondas")
          print(f"  Exitosos : {[a.id for a in exitosos]}")
          print(f"  Fallidos : {[a.id for a in fallidos]}")
          print(f"{'='*55}\n")

  return {
      "rondas_totales": ronda,
      "exitosos": len(exitosos),
      "fallidos": len(fallidos),
      "agentes": agentes,
  }


def experimento_estadistico(n_agentes: int, repeticiones: int = 1000) -> None:
    """
    Ejecuta múltiples simulaciones y muestra estadísticas promedio.
    """
    print(f"\n{'='*55}")
    print(f"  Experimento: {n_agentes} agentes, {repeticiones} repeticiones")
    print(f"{'='*55}")

    total_rondas = 0
    total_exitosos = 0

    for _ in range(repeticiones):
        agentes = [Agente(id=i) for i in range(n_agentes)]
        resultado = simular_canal(agentes, verbose=False)
        total_rondas   += resultado["rondas_totales"]
        total_exitosos += resultado["exitosos"]

    print(f"  Rondas promedio    : {total_rondas / repeticiones:.2f}")
    print(f"  Éxitos promedio    : {total_exitosos / repeticiones:.2f} / {n_agentes}")
    print(f"  Tasa de éxito      : {total_exitosos / (repeticiones * n_agentes) * 100:.1f}%")
    print()


# ── Ejecución principal ─────────────────────────────────────────────────────

if __name__ == "__main__":
    random.seed(42)

    # Ejemplo detallado con 4 agentes
    agentes = [Agente(id=i) for i in range(4)]
    simular_canal(agentes, verbose=True)

    # Estadísticas para distintas cantidades de agentes
    for n in [2, 4, 8, 16]:
        experimento_estadistico(n_agentes=n, repeticiones=2000)

Iniciando simulacion con 4 agente(s)

Ronda   1 | pendientes: [0, 1, 2, 3]
Esperas: {a.id: esperas[a.id] for a in pendientes}
Tranmiten: [0, 1, 2, 3] ->x colision entre [0, 1, 2, 3]

Ronda   2 | pendientes: [0, 1, 2, 3]
Esperas: {a.id: esperas[a.id] for a in pendientes}
Tranmiten: [0, 1, 2, 3] ->x colision entre [0, 1, 2, 3]

Ronda   3 | pendientes: [0, 1, 2, 3]
Esperas: {a.id: esperas[a.id] for a in pendientes}
Tranmiten: [1, 2, 3] ->x colision entre [1, 2, 3]

Ronda   4 | pendientes: [0, 1, 2, 3]
Esperas: {a.id: esperas[a.id] for a in pendientes}
Tranmiten: [2] ->Agente 2 transmite con exito (k=3)

Ronda   5 | pendientes: [0, 1, 3]
Esperas: {a.id: esperas[a.id] for a in pendientes}
Tranmiten: [] ->(nadie transmite)

Ronda   6 | pendientes: [0, 1, 3]
Esperas: {a.id: esperas[a.id] for a in pendientes}
Tranmiten: [1] ->Agente 1 transmite con exito (k=3)

Ronda   7 | pendientes: [0, 3]
Esperas: {a.id: esperas[a.id] for a in pendientes}
Tranmiten: [] ->(nadie transmite)

Ronda   8 | pendi